## 依存関係のインポート


In [ ]:
import os

from dotenv import load_dotenv
from openai import AzureOpenAI, OpenAI

## 環境変数の読み込み

In [ ]:
load_dotenv()

In [ ]:
resource_name = os.getenv("AZURE_OPENAI_API_RESOURCE_NAME")
key = os.getenv("AZURE_OPENAI_API_KEY")
deployment_name = "gpt-5"

if not resource_name or not key:
    raise ValueError(
        "Please set the AZURE_OPENAI_API_RESOURCE_NAME and AZURE_OPENAI_KEY in the .env file"
    )

## Responses APIによるモデル呼び出し

### クライアント初期化

In [ ]:
openai_client = OpenAI(
    api_key=key,
    base_url=f"https://{resource_name}.openai.azure.com/openai/v1",
)

### Responses APIの呼び出し

In [ ]:
response = openai_client.responses.create(
    model=deployment_name,
    instructions="You are a helpful assistant.",
    input="あなたは何ができますか？",
)

print(response.model_dump_json(indent=2))

### Responses APIによるコンテキストを引き継いだ2回目の呼び出し

In [ ]:
second_response = openai_client.responses.create(
    model=deployment_name,
    previous_response_id=response.id,
    input="では、英語で教えてください。",
)

print(second_response.model_dump_json(indent=2))

## Chat Completions APIによるモデル呼び出し

### クライアント初期化

In [ ]:
azure_openai_client = AzureOpenAI(
    api_version="2025-04-01-preview",
    azure_endpoint=f"https://{resource_name}.cognitiveservices.azure.com/",
    api_key=key,
)

### Chat Completions APIの呼び出し

In [ ]:
completion_response = azure_openai_client.chat.completions.create(
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "あなたは何ができますか？",
        },
    ],
    model=deployment_name,
)

print(completion_response.choices[0].message.content)